In [0]:
# Create a sample nested JSON file with various complexity levels
import json

# Complex nested JSON with multiple levels and arrays
data = [
    {
        "customer_id": 1,
        "name": "Alice Johnson",
        "contact": {
            "email": "alice@example.com",
            "phone": {
                "home": "555-1234",
                "mobile": "555-5678"
            }
        },
        "orders": [
            {
                "order_id": "A001",
                "date": "2026-01-15",
                "items": [
                    {"product": "Laptop", "quantity": 1, "price": 1200},
                    {"product": "Mouse", "quantity": 2, "price": 25}
                ],
                "total": 1250
            },
            {
                "order_id": "A002",
                "date": "2026-02-20",
                "items": [
                    {"product": "Keyboard", "quantity": 1, "price": 80}
                ],
                "total": 80
            }
        ],
        "preferences": {
            "newsletter": True,
            "notifications": {"email": True, "sms": False}
        }
    },
    {
        "customer_id": 2,
        "name": "Bob Smith",
        "contact": {
            "email": "bob@example.com",
            "phone": {
                "home": None,
                "mobile": "555-9999"
            }
        },
        "orders": [
            {
                "order_id": "B001",
                "date": "2026-03-10",
                "items": [
                    {"product": "Monitor", "quantity": 2, "price": 300}
                ],
                "total": 600
            }
        ],
        "preferences": {
            "newsletter": False,
            "notifications": {"email": False, "sms": True}
        }
    }
]

# Write to DBFS using dbutils
dbutils.fs.put("/FileStore/nested_customers.json", json.dumps(data, indent=2), overwrite=True)

print("✓ Created sample nested JSON file")
print(f"Sample data has {len(data)} customers with nested orders and contact info")

In [0]:
# Method 1: Read JSON file - PySpark automatically infers nested structure
df_nested = spark.read.json('/FileStore/nested_customers.json')

print("=== Original Nested Schema ===")
df_nested.printSchema()
print("\n=== Sample Data (nested) ===")
display(df_nested)

In [0]:
# Method 2: Use dot notation to access nested fields
from pyspark.sql.functions import col

df_flattened_basic = df_nested.select(
    col("customer_id"),
    col("name"),
    col("contact.email").alias("email"),
    col("contact.phone.home").alias("home_phone"),
    col("contact.phone.mobile").alias("mobile_phone"),
    col("preferences.newsletter").alias("newsletter_opt_in"),
    col("preferences.notifications.email").alias("email_notifications"),
    col("preferences.notifications.sms").alias("sms_notifications")
)

print("=== Flattened Schema (dot notation) ===")
df_flattened_basic.printSchema()
print("\n=== Flattened Data ===")
display(df_flattened_basic)

In [0]:
# Method 3: Use wildcard to flatten one level at a time
df_contact_expanded = df_nested.select(
    "customer_id",
    "name",
    "contact.*",  # Expands all fields within contact
    "preferences.*"  # Expands all fields within preferences
)

print("=== Schema after wildcard expansion ===")
df_contact_expanded.printSchema()
print("\n=== Data after wildcard expansion ===")
display(df_contact_expanded)

In [0]:
# Method 4: Use explode() to convert array elements into separate rows
from pyspark.sql.functions import explode, explode_outer

# Explode the orders array - creates one row per order
df_orders_exploded = df_nested.select(
    col("customer_id"),
    col("name"),
    col("contact.email").alias("email"),
    explode("orders").alias("order")  # Creates new row for each order
)

print("=== After exploding orders array ===")
df_orders_exploded.printSchema()
print("\n=== Exploded orders (one row per order) ===")
display(df_orders_exploded)

print(f"\nOriginal rows: {df_nested.count()}")
print(f"After explode: {df_orders_exploded.count()}")

In [0]:
# Method 5: Chain explode operations for deeply nested arrays
# First explode orders, then explode items within each order

df_fully_flattened = df_nested.select(
    col("customer_id"),
    col("name"),
    col("contact.email").alias("email"),
    explode("orders").alias("order")
).select(
    "customer_id",
    "name",
    "email",
    col("order.order_id"),
    col("order.date").alias("order_date"),
    col("order.total").alias("order_total"),
    explode("order.items").alias("item")  # Second explode for items
).select(
    "customer_id",
    "name",
    "email",
    "order_id",
    "order_date",
    "order_total",
    col("item.product"),
    col("item.quantity"),
    col("item.price")
)

print("=== Fully flattened (customer -> order -> item level) ===")
df_fully_flattened.printSchema()
print("\n=== One row per item in every order ===")
display(df_fully_flattened)

print(f"\nOriginal customers: {df_nested.count()}")
print(f"After full flattening: {df_fully_flattened.count()} rows (one per item)")

In [0]:
# Method 6: Use SQL with LATERAL VIEW EXPLODE for array flattening
df_nested.createOrReplaceTempView("customers_json")

df_sql_flattened = spark.sql("""
    SELECT 
        c.customer_id,
        c.name,
        c.contact.email as email,
        c.contact.phone.mobile as mobile_phone,
        o.order_id,
        o.date as order_date,
        o.total as order_total,
        i.product,
        i.quantity,
        i.price
    FROM customers_json c
    LATERAL VIEW EXPLODE(c.orders) orders_table as o
    LATERAL VIEW EXPLODE(o.items) items_table as i
""")

print("=== SQL-based flattening with LATERAL VIEW ===")
df_sql_flattened.printSchema()
print("\n=== Result ===")
display(df_sql_flattened)

In [0]:
# Method 7: When JSON data is stored as STRING column, use from_json()
from pyspark.sql.functions import from_json, to_json, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Simulate JSON string data
df_json_strings = spark.createDataFrame([
    (1, '{"name": "Product A", "specs": {"weight": 10, "color": "red"}}'),
    (2, '{"name": "Product B", "specs": {"weight": 20, "color": "blue"}}')
], ["id", "json_data"])

print("=== Original data with JSON strings ===")
display(df_json_strings)

# Define schema for the JSON string
json_schema = StructType([
    StructField("name", StringType()),
    StructField("specs", StructType([
        StructField("weight", IntegerType()),
        StructField("color", StringType())
    ]))
])

# Parse JSON strings into structured columns
df_parsed = df_json_strings.withColumn(
    "parsed", from_json(col("json_data"), json_schema)
).select(
    "id",
    col("parsed.name").alias("product_name"),
    col("parsed.specs.weight").alias("weight"),
    col("parsed.specs.color").alias("color")
)

print("\n=== After parsing JSON strings ===")
display(df_parsed)

In [0]:
# Method 8: Use higher-order functions (transform, filter, aggregate) on nested arrays
from pyspark.sql.functions import transform, filter as array_filter, expr, aggregate, array_max

# Transform items within orders (apply operation to each element)
df_with_discounted_prices = df_nested.withColumn(
    "orders_with_discount",
    transform(
        "orders",
        lambda o: o.withField(
            "items",
            transform(
                o.getField("items"),
                lambda i: i.withField("discounted_price", i.getField("price") * 0.9)
            )
        )
    )
)

print("=== Using transform() to add discounted prices ===")
display(df_with_discounted_prices.select(
    "customer_id", 
    "name", 
    "orders_with_discount"
))

# Filter items in orders (keep only items with price > 50)
df_filtered_items = df_nested.withColumn(
    "expensive_orders",
    transform(
        "orders",
        lambda o: o.withField(
            "items",
            array_filter(o.getField("items"), lambda i: i.getField("price") > 50)
        )
    )
)

print("\n=== Using filter() to keep only items with price > 50 ===")
display(df_filtered_items.select("customer_id", "name", "expensive_orders"))

In [0]:
# Method 9: Recursive function to completely flatten all nested structures
from pyspark.sql.types import StructType, ArrayType

def flatten_dataframe(df, separator="_"):
    """
    Recursively flatten all nested structures in a DataFrame.
    Converts StructType to flat columns with separator between parent and child names.
    Note: This does NOT explode arrays - it keeps them as array columns.
    """
    flat_cols = []
    
    def flatten_struct(schema, prefix=""):
        for field in schema.fields:
            col_name = f"{prefix}{separator}{field.name}" if prefix else field.name
            
            if isinstance(field.dataType, StructType):
                # Recursively flatten nested struct
                flatten_struct(field.dataType, col_name)
            else:
                # Add the column (leaf node or array)
                flat_cols.append(col(f"`{col_name.replace('.', separator)}`") if '.' in col_name else col(col_name).alias(col_name.replace('.', separator)))
    
    flatten_struct(df.schema)
    return df.select(*[col(c).alias(c.replace(".", separator)) for c in df.columns])

# Apply recursive flattening
df_recursive_flat = flatten_dataframe(df_nested)

print("=== Recursively flattened schema ===")
df_recursive_flat.printSchema()
print("\n=== Recursively flattened data ===")
display(df_recursive_flat.limit(1))

In [0]:
# Method 10: Modern read_files() approach and explode_outer for null safety

# Using read_files() - auto-detects format
df_read_files = spark.read.format("json").load('/FileStore/nested_customers.json')
print("=== Using read_files (auto-format detection) ===")
print(f"Read {df_read_files.count()} records\n")

# explode_outer vs explode: explode_outer keeps rows even when array is null/empty
from pyspark.sql.functions import explode_outer

# Create test data with null array
test_data = [
    {"id": 1, "name": "Alice", "orders": [{"order_id": "A1"}]},
    {"id": 2, "name": "Bob", "orders": None},  # Null array
    {"id": 3, "name": "Carol", "orders": []}   # Empty array
]

import json
dbutils.fs.put("/FileStore/test_nulls.json", json.dumps(test_data), overwrite=True)

df_test = spark.read.json('/FileStore/test_nulls.json')

print("=== Comparison: explode vs explode_outer ===")
print("\nUsing explode() - loses rows with null/empty arrays:")
df_explode = df_test.select("id", "name", explode("orders").alias("order"))
display(df_explode)
print(f"Rows after explode: {df_explode.count()}")

print("\nUsing explode_outer() - keeps rows with null/empty arrays:")
df_explode_outer = df_test.select("id", "name", explode_outer("orders").alias("order"))
display(df_explode_outer)
print(f"Rows after explode_outer: {df_explode_outer.count()}")

## 📋 Summary: JSON Flattening Methods

### **Simple Nested Objects (no arrays)**
* **Dot notation**: `col("contact.email")` - Best for direct field access
* **Wildcard**: `select("contact.*")` - Expands one struct level

### **Arrays in JSON**
* **explode()**: Convert array to rows - use when you want one row per array element
* **explode_outer()**: Same as explode but keeps null/empty arrays - safer option
* **LATERAL VIEW EXPLODE**: SQL equivalent for array flattening

### **Deeply Nested Arrays**
* **Chain explode()**: Multiple explode operations for multi-level arrays (orders → items)
* **Higher-order functions**: `transform()`, `filter()`, `aggregate()` - transform arrays without exploding

### **JSON Strings (not structured)**
* **from_json()**: Parse JSON strings into structured columns with schema
* **get_json_object()**: Extract specific values using JSONPath

### **Complex/Dynamic Structures**
* **Recursive flattening function**: Fully flatten all nested structs automatically
* **Custom logic**: Combine multiple techniques for specific needs

### **Reading JSON Files**
* **spark.read.json()**: Standard PySpark JSON reader
* **read_files()**: Modern Databricks approach with auto-format detection
* **Options**: `multiLine=true` for pretty-printed JSON, `dateFormat`, `timestampFormat`

### **Best Practices**
1. ✅ Use `explode_outer()` instead of `explode()` to avoid losing rows
2. ✅ Apply `explode()` early in transformation pipeline for better performance
3. ✅ Use explicit schema with `from_json()` for better type control
4. ✅ Consider `transform()` for array operations without row explosion
5. ✅ Check schema with `printSchema()` to understand nested structure before flattening

---

# 🔍 Schema Discovery & Definition Methods

Comprehensive guide to finding, defining, and inspecting schemas in PySpark

In [0]:
# Method 1: Automatic Schema Inference
# PySpark automatically infers schema from data

import json
from pyspark.sql.types import *

# Create sample JSON data directly in memory
json_data = [
    '{"id": 1, "name": "Alice", "age": 30, "salary": 75000.50, "is_active": true, "join_date": "2020-01-15"}',
    '{"id": 2, "name": "Bob", "age": 25, "salary": 65000.00, "is_active": false, "join_date": "2021-03-20"}',
    '{"id": 3, "name": "Carol", "age": 35, "salary": 85000.75, "is_active": true, "join_date": "2019-07-10"}'
]

# Create DataFrame from JSON strings - schema is automatically inferred
df_auto = spark.read.json(spark.sparkContext.parallelize(json_data))

print("=== Automatically Inferred Schema ===")
df_auto.printSchema()
print("\n=== Data Types ===")
for col_name, dtype in df_auto.dtypes:
    print(f"{col_name}: {dtype}")

display(df_auto)

In [0]:
# Method 2: Define Schema Explicitly using StructType and StructField
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType, DateType

# Define explicit schema
explicit_schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("name", StringType(), nullable=False),
    StructField("age", IntegerType(), nullable=True),
    StructField("salary", DoubleType(), nullable=True),
    StructField("is_active", BooleanType(), nullable=True),
    StructField("join_date", StringType(), nullable=True)  # Can use DateType() too
])

print("=== Explicitly Defined Schema ===")
print(explicit_schema)
print("\nSchema as DDL string:")
print(explicit_schema.simpleString())

# Read data with explicit schema
df_explicit = spark.read.schema(explicit_schema).json(spark.sparkContext.parallelize(json_data))

print("\n=== DataFrame with Explicit Schema ===")
df_explicit.printSchema()
display(df_explicit)

In [0]:
# Method 3: Define Schema using DDL (Data Definition Language) String
# Simpler syntax, great for quick prototypes

ddl_schema = "id INT, name STRING, age INT, salary DOUBLE, is_active BOOLEAN, join_date STRING"

print("=== DDL String Schema ===")
print(ddl_schema)

# Read with DDL schema
df_ddl = spark.read.schema(ddl_schema).json(spark.sparkContext.parallelize(json_data))

print("\n=== DataFrame with DDL Schema ===")
df_ddl.printSchema()

# You can also convert between DDL and StructType
print("\nConvert StructType to DDL:")
print(df_ddl.schema.simpleString())

# Parse DDL string to StructType
from pyspark.sql.types import _parse_datatype_string
parsed_schema = _parse_datatype_string(ddl_schema)
print("\nParsed DDL to StructType:")
print(parsed_schema)

In [0]:
# Method 4: Different ways to inspect existing schemas

df = df_auto  # Use the auto-inferred DataFrame

print("=== 1. printSchema() - Tree format ===")
df.printSchema()

print("\n=== 2. schema property - Returns StructType ===")
print(df.schema)

print("\n=== 3. dtypes - List of (name, type) tuples ===")
print(df.dtypes)

print("\n=== 4. columns - List of column names ===")
print(df.columns)

print("\n=== 5. schema.fields - List of StructField objects ===")
for field in df.schema.fields:
    print(f"Field: {field.name}, Type: {field.dataType}, Nullable: {field.nullable}")

print("\n=== 6. schema.fieldNames() - Column names ===")
print(df.schema.fieldNames())

print("\n=== 7. schema.simpleString() - DDL format ===")
print(df.schema.simpleString())

print("\n=== 8. schema.jsonValue() - JSON representation ===")
import json
print(json.dumps(df.schema.jsonValue(), indent=2))

In [0]:
# Method 5: Use schema_of_json() to infer schema from a JSON sample
from pyspark.sql.functions import schema_of_json, lit

# Sample JSON string
sample_json = '{"customer": {"id": 123, "name": "John", "contact": {"email": "john@example.com", "phone": "555-1234"}}}'

# Infer schema from JSON string
schema_expr = schema_of_json(lit(sample_json))

print("=== Schema inferred from JSON string ===")
df_schema_check = spark.range(1).select(schema_expr.alias("schema"))
display(df_schema_check)

# Use the inferred schema to parse JSON data
from pyspark.sql.functions import from_json

json_strings_df = spark.createDataFrame([
    (1, '{"customer": {"id": 123, "name": "John", "contact": {"email": "john@example.com", "phone": "555-1234"}}}'),
    (2, '{"customer": {"id": 456, "name": "Jane", "contact": {"email": "jane@example.com", "phone": "555-5678"}}}')
], ["row_id", "json_data"])

# Get the schema as a string first
schema_string = df_schema_check.collect()[0][0]
print(f"\nInferred schema DDL: {schema_string}")

# Parse using from_json with the schema
from pyspark.sql.types import _parse_datatype_string
inferred_struct = _parse_datatype_string(schema_string)

df_parsed = json_strings_df.withColumn("parsed", from_json("json_data", inferred_struct))
print("\n=== Parsed data using inferred schema ===")
df_parsed.select("row_id", "parsed.*").printSchema()
display(df_parsed.select("row_id", "parsed.*"))

In [0]:
# Method 6: Define complex nested schemas with StructType, ArrayType, MapType
from pyspark.sql.types import ArrayType, MapType

# Complex nested schema definition
complex_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), False),
    # Nested struct
    StructField("contact", StructType([
        StructField("email", StringType(), True),
        StructField("phones", ArrayType(StringType()), True),  # Array of strings
        StructField("address", StructType([
            StructField("street", StringType(), True),
            StructField("city", StringType(), True),
            StructField("zip", StringType(), True)
        ]), True)
    ]), True),
    # Array of structs
    StructField("orders", ArrayType(StructType([
        StructField("order_id", StringType(), False),
        StructField("amount", DoubleType(), False),
        StructField("items", ArrayType(StringType()), True)
    ])), True),
    # Map type
    StructField("metadata", MapType(StringType(), StringType()), True)
])

print("=== Complex Nested Schema ===")
print(complex_schema.treeString())

print("\n=== DDL Format ===")
print(complex_schema.simpleString())

# Create sample data matching this schema
sample_complex_data = [
    {
        "customer_id": 1,
        "name": "Alice",
        "contact": {
            "email": "alice@example.com",
            "phones": ["555-1234", "555-5678"],
            "address": {"street": "123 Main St", "city": "NYC", "zip": "10001"}
        },
        "orders": [
            {"order_id": "A001", "amount": 100.50, "items": ["laptop", "mouse"]},
            {"order_id": "A002", "amount": 50.25, "items": ["keyboard"]}
        ],
        "metadata": {"source": "web", "campaign": "summer_sale"}
    }
]

df_complex = spark.createDataFrame(sample_complex_data, schema=complex_schema)
print("\n=== DataFrame with Complex Schema ===")
df_complex.printSchema()
display(df_complex)

In [0]:
# Method 7: Discover schema from existing Delta/Parquet/Tables

# Create a sample table first
df_sample = spark.createDataFrame([
    (1, "Product A", 100.50, "Electronics"),
    (2, "Product B", 250.75, "Furniture"),
    (3, "Product C", 75.00, "Electronics")
], ["product_id", "product_name", "price", "category"])

# Save as temporary view
df_sample.createOrReplaceTempView("products_temp")

print("=== Method 7a: Get schema from table using spark.table() ===")
df_from_table = spark.table("products_temp")
df_from_table.printSchema()

print("\n=== Method 7b: Using DESCRIBE command (SQL) ===")
describe_result = spark.sql("DESCRIBE products_temp")
display(describe_result)

print("\n=== Method 7c: DESCRIBE EXTENDED for detailed info ===")
describe_extended = spark.sql("DESCRIBE EXTENDED products_temp")
display(describe_extended)

print("\n=== Method 7d: Extract and reuse schema ===")
# Get schema from existing DataFrame and use it for new data
reused_schema = df_from_table.schema
print("Reused schema:")
print(reused_schema)

# Apply to new data
new_data = [(4, "Product D", 150.00, "Clothing")]
df_new = spark.createDataFrame(new_data, schema=reused_schema)
print("\nNew DataFrame with reused schema:")
df_new.printSchema()

In [0]:
# Method 8: Merge schemas from multiple DataFrames

# Create two DataFrames with different schemas
df1 = spark.createDataFrame([
    (1, "Alice", 30),
    (2, "Bob", 25)
], ["id", "name", "age"])

df2 = spark.createDataFrame([
    (3, "Carol", "Engineering", 75000),
    (4, "David", "Sales", 65000)
], ["id", "name", "department", "salary"])

print("=== DataFrame 1 Schema ===")
df1.printSchema()

print("\n=== DataFrame 2 Schema ===")
df2.printSchema()

print("\n=== Method 8a: Union by name (fills missing columns with null) ===")
# unionByName handles different schemas
df_union = df1.unionByName(df2, allowMissingColumns=True)
df_union.printSchema()
display(df_union)

print("\n=== Method 8b: Merge schemas manually ===")
# Get all unique column names from both DataFrames
all_columns = list(set(df1.columns + df2.columns))
print(f"All columns: {all_columns}")

# Add missing columns to each DataFrame
from pyspark.sql.functions import lit

for col_name in all_columns:
    if col_name not in df1.columns:
        df1 = df1.withColumn(col_name, lit(None))
    if col_name not in df2.columns:
        df2 = df2.withColumn(col_name, lit(None))

# Now union with matching schemas
df_merged = df1.union(df2)
print("\nMerged schema:")
df_merged.printSchema()
display(df_merged)

In [0]:
# Method 9: Modify schemas - add, rename, drop, cast columns

df_original = spark.createDataFrame([
    (1, "Alice", "30", "2020-01-15"),
    (2, "Bob", "25", "2021-03-20")
], ["id", "name", "age", "join_date"])

print("=== Original Schema ===")
df_original.printSchema()

print("\n=== 9a: Add new columns ===")
from pyspark.sql.functions import current_timestamp, lit
df_with_cols = df_original.withColumn("status", lit("active")) \
                          .withColumn("created_at", current_timestamp())
df_with_cols.printSchema()

print("\n=== 9b: Rename columns ===")
df_renamed = df_original.withColumnRenamed("name", "employee_name") \
                       .withColumnRenamed("age", "employee_age")
df_renamed.printSchema()

print("\n=== 9c: Drop columns ===")
df_dropped = df_original.drop("age")
df_dropped.printSchema()

print("\n=== 9d: Cast/Change column types ===")
from pyspark.sql.functions import col
df_casted = df_original.withColumn("age", col("age").cast(IntegerType())) \
                      .withColumn("join_date", col("join_date").cast(DateType()))
df_casted.printSchema()
print("Data with corrected types:")
display(df_casted)

print("\n=== 9e: Reorder columns ===")
df_reordered = df_original.select("id", "join_date", "name", "age")
df_reordered.printSchema()

print("\n=== 9f: Select and cast multiple columns at once ===")
df_transformed = df_original.selectExpr(
    "id",
    "name",
    "CAST(age AS INT) as age",
    "TO_DATE(join_date) as join_date",
    "'active' as status"
)
df_transformed.printSchema()
display(df_transformed)

In [0]:
# Method 10: Validate and compare schemas

# Create schemas for comparison
schema1 = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

schema2 = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("salary", DoubleType(), True)  # Different field
])

print("=== 10a: Compare two schemas ===")
print(f"Schemas are equal: {schema1 == schema2}")
print(f"Schema1 fields: {[f.name for f in schema1.fields]}")
print(f"Schema2 fields: {[f.name for f in schema2.fields]}")

print("\n=== 10b: Find schema differences ===")
schema1_fields = set(f.name for f in schema1.fields)
schema2_fields = set(f.name for f in schema2.fields)

print(f"Fields only in schema1: {schema1_fields - schema2_fields}")
print(f"Fields only in schema2: {schema2_fields - schema1_fields}")
print(f"Common fields: {schema1_fields & schema2_fields}")

print("\n=== 10c: Validate data against schema ===")
# Try to create DataFrame with mismatched data
try:
    # This will fail - age as string instead of int
    bad_data = [(1, "Alice", "thirty")]  # age should be int
    df_bad = spark.createDataFrame(bad_data, schema=schema1)
    print("Data validated successfully")
except Exception as e:
    print(f"Validation failed: {str(e)[:100]}...")

# Correct data
good_data = [(1, "Alice", 30)]
df_good = spark.createDataFrame(good_data, schema=schema1)
print("\nCorrect data validated successfully")
df_good.printSchema()

print("\n=== 10d: Check if column exists ===")
df_test = spark.range(1).select(lit(1).alias("id"), lit("test").alias("name"))
print(f"Has 'id' column: {'id' in df_test.columns}")
print(f"Has 'age' column: {'age' in df_test.columns}")

print("\n=== 10e: Get schema field by name ===")
try:
    field = schema1["name"]  # Access field by name
    print(f"Field 'name': type={field.dataType}, nullable={field.nullable}")
except KeyError:
    print("Field not found")

In [0]:
# Method 11: Use sampling to infer schema from large datasets

print("=== 11a: Sample data to understand schema quickly ===")
# For large files, you can sample to infer schema faster
large_data = [(i, f"User_{i}", 20 + i % 40, i * 1000.0) for i in range(1, 1001)]
df_large = spark.createDataFrame(large_data, ["id", "name", "age", "value"])

print(f"Total rows: {df_large.count()}")
print("\nSample 5 rows to understand data:")
display(df_large.limit(5))

print("\n=== 11b: Infer schema with samplingRatio ===")
# When reading large CSV/JSON files, use samplingRatio for faster schema inference
print("Reading with samplingRatio=0.1 (10% of data) for schema inference")
print("This is useful for very large files where full scan is expensive")

# Example: spark.read.option("samplingRatio", 0.1).csv("large_file.csv")

print("\n=== 11c: Discover data types from sample ===")
from pyspark.sql.functions import col

# Analyze a sample to understand type patterns
df_sample_analysis = df_large.limit(10)
print("\nSample data types:")
for field in df_sample_analysis.schema.fields:
    sample_values = df_sample_analysis.select(field.name).limit(3).collect()
    print(f"{field.name}: {field.dataType} - Examples: {[row[0] for row in sample_values]}")

print("\n=== 11d: Profile data to refine schema ===")
# Get min, max, distinct count to understand if type is appropriate
df_profile = df_large.select(
    col("age").cast("string").alias("column"),
    lit("age").alias("column_name")
).union(
    df_large.select(
        col("value").cast("string").alias("column"),
        lit("value").alias("column_name")
    )
)

print("\nData profiling to validate schema choices:")
df_summary = df_large.select(
    lit("age").alias("column"),
    col("age").cast("string").alias("min_value"),
    col("age").cast("string").alias("max_value")
).limit(1)

from pyspark.sql.functions import min, max, count, countDistinct
age_stats = df_large.agg(
    min("age").alias("min_age"),
    max("age").alias("max_age"),
    count("age").alias("count"),
    countDistinct("age").alias("distinct_count")
)
print("\nAge column statistics:")
display(age_stats)

---

## 📚 Complete Schema Methods Reference

### **Discovery Methods (Finding Schemas)**

| Method | Use Case | Code Example |
|--------|----------|-------------|
| **Automatic Inference** | Let PySpark figure it out | `spark.read.json(path)` |
| **printSchema()** | View schema in tree format | `df.printSchema()` |
| **df.schema** | Get StructType object | `schema = df.schema` |
| **df.dtypes** | Get list of (name, type) tuples | `types = df.dtypes` |
| **df.columns** | Get column names list | `cols = df.columns` |
| **schema.fields** | Access StructField objects | `for field in df.schema.fields: ...` |
| **schema_of_json()** | Infer from JSON string | `schema_of_json(lit(json_str))` |
| **spark.table()** | Get schema from existing table | `df = spark.table("table_name")` |
| **DESCRIBE** | SQL-based schema inspection | `DESCRIBE table_name` |
| **samplingRatio** | Fast inference on large files | `.option("samplingRatio", 0.1)` |

### **Definition Methods (Creating Schemas)**

| Method | Use Case | Code Example |
|--------|----------|-------------|
| **StructType/StructField** | Programmatic definition | `StructType([StructField("id", IntegerType())])` |
| **DDL String** | Quick text-based definition | `"id INT, name STRING, age INT"` |
| **From existing DataFrame** | Reuse schema | `new_df = spark.createDataFrame(data, df.schema)` |
| **Nested StructType** | Complex nested objects | `StructField("address", StructType([...]))` |
| **ArrayType** | Array columns | `StructField("tags", ArrayType(StringType()))` |
| **MapType** | Key-value columns | `StructField("meta", MapType(StringType(), StringType()))` |

### **Modification Methods (Changing Schemas)**

| Method | Use Case | Code Example |
|--------|----------|-------------|
| **withColumn()** | Add/modify column | `df.withColumn("new_col", lit("value"))` |
| **withColumnRenamed()** | Rename column | `df.withColumnRenamed("old", "new")` |
| **drop()** | Remove column | `df.drop("column_name")` |
| **cast()** | Change type | `df.withColumn("age", col("age").cast(IntegerType()))` |
| **select()** | Reorder/subset columns | `df.select("col1", "col3", "col2")` |
| **selectExpr()** | Transform with SQL | `df.selectExpr("CAST(age AS INT)")` |

### **Validation & Comparison Methods**

| Method | Use Case | Code Example |
|--------|----------|-------------|
| **schema1 == schema2** | Compare schemas | `if df1.schema == df2.schema: ...` |
| **field in columns** | Check column exists | `if "age" in df.columns: ...` |
| **schema["name"]** | Get field by name | `field = schema["id"]` |
| **createDataFrame with schema** | Validate data | `spark.createDataFrame(data, schema)` |
| **unionByName** | Merge different schemas | `df1.unionByName(df2, allowMissingColumns=True)` |

### **Conversion Methods**

| Method | Use Case | Code Example |
|--------|----------|-------------|
| **simpleString()** | StructType → DDL | `schema.simpleString()` |
| **jsonValue()** | StructType → JSON | `schema.jsonValue()` |
| **treeString()** | StructType → Tree format | `schema.treeString()` |
| **_parse_datatype_string()** | DDL → StructType | `_parse_datatype_string(ddl_str)` |

### **Common Data Types Reference**

```python
# Numeric Types
ByteType()         # 8-bit integer
ShortType()        # 16-bit integer
IntegerType()      # 32-bit integer
LongType()         # 64-bit integer
FloatType()        # 32-bit float
DoubleType()       # 64-bit float
DecimalType(p, s)  # Decimal with precision and scale

# String & Binary
StringType()       # String
BinaryType()       # Binary data

# Boolean
BooleanType()      # True/False

# Date & Time
DateType()         # Date (year, month, day)
TimestampType()    # Timestamp (date + time)
TimestampNTZType() # Timestamp without timezone

# Complex Types
ArrayType(elementType)              # Array of elements
MapType(keyType, valueType)         # Key-value map
StructType([StructField(...), ...]) # Nested structure
```

### **Best Practices**

1. ✅ **Use explicit schemas for production** - Faster and prevents type inference errors
2. ✅ **DDL strings for quick prototypes** - Easier to read and write
3. ✅ **Validate schemas early** - Catch type mismatches before processing large data
4. ✅ **Use nullable=False for required fields** - Better data quality enforcement
5. ✅ **Sample large files** - Use samplingRatio for faster schema discovery
6. ✅ **Reuse schemas** - Extract schema once and apply to similar data
7. ✅ **Document complex schemas** - Add comments explaining nested structures
8. ⚠️ **Avoid frequent schema operations in loops** - Cache schema outside loops
9. ⚠️ **Be careful with type inference** - May guess wrong types (e.g., int vs long)
10. ⚠️ **Test schema changes** - Breaking changes can fail downstream jobs